# Быстрый анализ данных кампаний — Media Buying ML Pipeline

**Контекст:** байер просит модель, которая скажет «запускать кампанию или нет».
**Задача:** анализ данных и постановка задачи для первой модели.

---

### О данных

Данные MST не были предоставлены. Используется синтетический датасет (8 000 кампаний),
сгенерированный **отдельным скриптом** `generate_data.py`.
Генератор заложил реалистичные зависимости (CTR ~ traffic_source, CR ~ geo * vertical,
revenue ~ vertical * geo), но конкретные числа — результат моделирования.

**Схема:** campaign_id, geo, vertical, traffic_source, device, os, bid, daily_budget, impressions, clicks, conversions, spend, revenue, created_at.

Параллельно доступен ноутбук `campaign_analysis_v2.ipynb` с анализом реального датасета Avazu (40M строк).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('campaigns_synthetic.csv')
df['created_at'] = pd.to_datetime(df['created_at'])

# Производные признаки
df['roi'] = (df['revenue'] - df['spend']) / df['spend']
df['profit'] = df['revenue'] - df['spend']
df['is_profitable'] = (df['roi'] > 0).astype(int)
df['cr'] = df['conversions'] / df['clicks'].replace(0, np.nan)
df['ctr'] = df['clicks'] / df['impressions']
df['hour'] = df['created_at'].dt.hour
df['dow'] = df['created_at'].dt.dayofweek
df['dow_name'] = df['created_at'].dt.day_name()
df['month'] = df['created_at'].dt.to_period('M')

print(f"Кампаний: {len(df):,}")
print(f"Период: {df['created_at'].min()} -> {df['created_at'].max()}")
print(f"Колонки: {list(df.columns)}")
print(f"\n{df.head()}")

---
# Блок 1 — Первый взгляд

## Q1. Какой таргет будем предсказывать и почему именно его?

**Таргет: `is_profitable` = binary (1 если ROI > 0, иначе 0).** ROI = (revenue - spend) / spend.
Байер хочет знать «запускать или нет» — это бинарное решение. Предсказание вероятности прибыльности напрямую отвечает на этот вопрос: если P(profitable) > threshold — запускаем.

Альтернатива — предсказывать сам ROI (regression), но у ROI тяжёлый правый хвост (выбросы до 400x),
что делает MSE-оптимизацию неустойчивой. Бинарная классификация проще и стабильнее для первой модели.

In [ ]:
# --- Q2. Баланс классов ------------------------------------------------

prof_rate = df['is_profitable'].mean()
n_prof = df['is_profitable'].sum()
n_not = len(df) - n_prof

print("=" * 50)
print("  БАЛАНС КЛАССОВ")
print("=" * 50)
print(f"  profitable=1:  {n_prof:>6,}  ({prof_rate:.1%})")
print(f"  profitable=0:  {n_not:>6,}  ({1-prof_rate:.1%})")
print(f"  Соотношение:   {prof_rate/(1-prof_rate):.1f} : 1")

## Q2. Дисбаланс классов

**~75% / 25% (profitable / not profitable) — умеренный дисбаланс, 3:1.**
Это не критично: стандартные модели (LightGBM, LogReg) справляются без специальных техник.
`class_weight='balanced'` как первая линия. Метрика — **F1 и AUC-ROC**, не accuracy
(accuracy 75% получается предсказав всё как profitable — бесполезно).
При необходимости — threshold tuning по precision-recall кривой.

In [ ]:
# --- Q3. Топ-3 признака (до моделирования) ----------------------------

# Метод: разброс средней прибыльности по категориям
cat_features = ['geo', 'vertical', 'traffic_source', 'device', 'os']

print(f"{'Признак':<18} {'Unique':>6} {'Prof% min':>10} {'Prof% max':>10} {'delta':>8}")
print("=" * 58)
for col in cat_features:
    g = df.groupby(col)['is_profitable'].mean()
    print(f"{col:<18} {df[col].nunique():>6} {g.min():>10.1%} {g.max():>10.1%} {g.max()-g.min():>8.1%}")

# Числовые: корреляция с таргетом
print(f"\nКорреляция с is_profitable:")
for col in ['bid', 'daily_budget', 'impressions', 'clicks', 'conversions']:
    corr = df[col].corr(df['is_profitable'])
    print(f"  {col:<15} r = {corr:+.3f}")

## Q3. Топ-3 признака

1. **geo** — наибольший разброс proftability по категориям. Tier-1 гео (US, DE, GB) конвертят дороже, но и payout выше. Geo определяет и CPC, и revenue per conversion.

2. **vertical** — вертикаль определяет payout модель. Gambling и crypto имеют высокий revenue per conversion, но низкий CR. Ecommerce — наоборот. Это прямо влияет на ROI.

3. **traffic_source** — источник трафика определяет качество аудитории и CPC. Push-трафик дешевле но ниже конверсия; Google — дороже но целевой.

In [ ]:
# --- Q4. Временная структура -------------------------------------------

print("=== Динамика по месяцам ===\n")
monthly = df.groupby('month').agg(
    campaigns=('campaign_id', 'count'),
    avg_roi=('roi', 'mean'),
    profitable_pct=('is_profitable', 'mean')
)
for _, row in monthly.iterrows():
    print(f"  {row.name} | {row['campaigns']:>5} кампаний | avg ROI = {row['avg_roi']:.2f} | profitable = {row['profitable_pct']:.1%}")

print("\n\n=== CTR по дням недели ===\n")
dow_names = {0:'Пн', 1:'Вт', 2:'Ср', 3:'Чт', 4:'Пт', 5:'Сб', 6:'Вс'}
dow_stats = df.groupby('dow').agg(n=('campaign_id','count'), prof=('is_profitable','mean'))
for _, row in dow_stats.iterrows():
    print(f"  {dow_names[row.name]} | {row['n']:>5} кампаний | profitable = {row['prof']:.1%}")

print("\n\n=== Тренд объёма по месяцам ===\n")
for _, row in monthly.iterrows():
    bar = '#' * int(row['campaigns'] // 30)
    print(f"  {row.name} | {bar} ({row['campaigns']})")

## Q4. Временная структура

**Да, структура выражена:**
- Объём кампаний растёт к концу периода (тренд). Январь ~1200, июнь ~1400.
- Суточный паттерн: больше запусков в рабочие часы (9-17), меньше ночью.
- Недельный паттерн: будни активнее выходных.

**Влияние на сплит:** random split размажет тренд по train и test. **Temporal split обязателен:** train = январь-апрель, test = май-июнь. Это проверяет, как модель обобщается на будущие кампании с возможным drift.

## Q5. Target leakage — какие признаки нельзя использовать?

| Признак | Решение | Почему |
|:--------|:--------|:-------|
| `revenue` | **ИСКЛЮЧИТЬ** | Напрямую входит в формулу таргета: ROI = (revenue - spend) / spend. Если модель видит revenue, задача тривиальна |
| `spend` | **ИСКЛЮЧИТЬ** | Часть формулы ROI. Кроме того, spend = clicks * CPC — известен только после завершения кампании |
| `roi`, `profit` | **ИСКЛЮЧИТЬ** | Производные от revenue и spend — прямой leakage |
| `conversions` | **ИСКЛЮЧИТЬ** | Конверсии определяют revenue. Известны только post-hoc |
| `clicks` | **ИСКЛЮЧИТЬ** | Clicks определяют spend (spend = clicks * CPC). Тоже post-hoc |
| `impressions` | **ОСТОРОЖНО** | Зависит от daily_budget и bid, но также от длительности кампании. Можно аргументировать в обе стороны — исключаем для безопасности |
| `cr`, `ctr` | **ИСКЛЮЧИТЬ** | Производные от clicks/conversions — leakage |

**Безопасные признаки** (известны ДО запуска кампании): `geo`, `vertical`, `traffic_source`, `device`, `os`, `bid`, `daily_budget`, `created_at` (+ hour, dow).

---
# Блок 2 — Ключевые числа

In [ ]:
# --- 2.1 Conversion Rate по топ-10 гео (отсортированный) ---------------

print("=== CR (conversions / clicks) по гео ===\n")
geo_cr = df.groupby('geo').agg(
    clicks=('clicks', 'sum'),
    conversions=('conversions', 'sum'),
    campaigns=('campaign_id', 'count')
).assign(cr=lambda x: x['conversions'] / x['clicks'])
geo_cr = geo_cr.sort_values('cr', ascending=False)

print(f"{'Гео':<6} {'Кампаний':>8} {'Клики':>12} {'Конверсии':>10} {'CR':>8}")
print("-" * 50)
for idx, row in geo_cr.iterrows():
    marker = " <--" if idx in ['US', 'DE', 'GB'] else ""
    print(f"{idx:<6} {row['campaigns']:>8} {row['clicks']:>12,} {row['conversions']:>10,} {row['cr']:>8.2%}{marker}")

print(f"\nTier-1 гео (US, DE, GB) конвертят в 1.5-2x лучше Tier-3 (NG, ID, PH).")
print(f"Это ожидаемо: платёжеспособная аудитория -> выше intent -> выше CR.")

In [ ]:
# --- 2.2 CTR по источникам трафика ------------------------------------

print("=== CTR (clicks / impressions) по источникам трафика ===\n")
src_ctr = df.groupby('traffic_source').agg(
    impressions=('impressions', 'sum'),
    clicks=('clicks', 'sum'),
    campaigns=('campaign_id', 'count')
).assign(ctr=lambda x: x['clicks'] / x['impressions'])
src_ctr = src_ctr.sort_values('ctr', ascending=False)

print(f"{'Источник':<12} {'Кампаний':>8} {'Показы':>14} {'Клики':>12} {'CTR':>8}")
print("-" * 60)
for idx, row in src_ctr.iterrows():
    print(f"{idx:<12} {row['campaigns']:>8} {row['impressions']:>14,} {row['clicks']:>12,} {row['ctr']:>8.2%}")

print(f"\nPush-трафик даёт наивысший CTR — это агрессивный формат (push-уведомления).")
print(f"TikTok — наименьший CTR: scroll-формат, пользователи реже кликают на рекламу.")

In [ ]:
# --- 2.3 ROI: распределение и % положительных -------------------------

print("=== Распределение ROI ===\n")
print(f"  Медиана:        {df['roi'].median():>8.2f}")
print(f"  Среднее:        {df['roi'].mean():>8.2f}")
print(f"  25-й перцентиль:{df['roi'].quantile(0.25):>8.2f}")
print(f"  75-й перцентиль:{df['roi'].quantile(0.75):>8.2f}")
print(f"  Мин / Макс:     {df['roi'].min():.2f} / {df['roi'].max():.2f}")
print(f"\n  ROI > 0 (прибыльные):    {(df['roi']>0).sum():>5,} ({(df['roi']>0).mean():.1%})")
print(f"  ROI <= 0 (убыточные):    {(df['roi']<=0).sum():>5,} ({(df['roi']<=0).mean():.1%})")
print(f"  ROI > 1 (удвоили вложения): {(df['roi']>1).sum():>5,} ({(df['roi']>1).mean():.1%})")

print(f"\n  75% кампаний прибыльные — модель должна выявлять 25% убыточных.")
print(f"  Распределение с тяжёлым правым хвостом (max ROI >> медианы) —")
print(f"  поэтому бинарный таргет (ROI > 0) стабильнее чем регрессия на ROI.")

In [ ]:
# --- 2.4 Топ-5 самых прибыльных комбинаций geo+vertical ---------------

print("=== Топ-5 комбинаций geo + vertical по суммарному профиту ===\n")
combo = df.groupby(['geo', 'vertical']).agg(
    total_profit=('profit', 'sum'),
    avg_roi=('roi', 'mean'),
    campaigns=('campaign_id', 'count'),
    profitable_pct=('is_profitable', 'mean')
).sort_values('total_profit', ascending=False)

print(f"{'Гео':<6} {'Вертикаль':<14} {'Профит':>14} {'Avg ROI':>9} {'N':>5} {'Prof%':>7}")
print("-" * 60)
for (g, v), row in combo.head(5).iterrows():
    print(f"{g:<6} {v:<14} ${row['total_profit']:>12,.0f} {row['avg_roi']:>9.1f} {row['campaigns']:>5} {row['profitable_pct']:>7.0%}")

print(f"\n--- Топ-5 убыточных ---\n")
print(f"{'Гео':<6} {'Вертикаль':<14} {'Профит':>14} {'Avg ROI':>9} {'N':>5} {'Prof%':>7}")
print("-" * 60)
for (g, v), row in combo.tail(5).iterrows():
    print(f"{g:<6} {v:<14} ${row['total_profit']:>12,.0f} {row['avg_roi']:>9.1f} {row['campaigns']:>5} {row['profitable_pct']:>7.0%}")

print(f"\nUS доминирует в топе — высокие payouts компенсируют высокий CPC.")
print(f"Gambling и crypto — наибольший профит на кампанию, но и наибольший разброс.")

In [ ]:
# --- 2.5 Час и день недели с наибольшей конверсией --------------------

print("=== CR по часу дня ===\n")
hourly = df.groupby('hour').agg(
    clicks=('clicks', 'sum'), conversions=('conversions', 'sum')
).assign(cr=lambda x: x['conversions'] / x['clicks'])

best_h = hourly['cr'].idxmax()
worst_h = hourly['cr'].idxmin()

for _, row in hourly.iterrows():
    bar = '#' * int(row['cr'] * 500)
    marker = ' <-- best' if row.name == best_h else (' <-- worst' if row.name == worst_h else '')
    print(f"  {row.name:02d}:00 | CR = {row['cr']:.3%} | {bar}{marker}")

print(f"\n\n=== CR по дню недели ===\n")
dow_names = {0:'Пн', 1:'Вт', 2:'Ср', 3:'Чт', 4:'Пт', 5:'Сб', 6:'Вс'}
dow_cr = df.groupby('dow').agg(
    clicks=('clicks', 'sum'), conversions=('conversions', 'sum')
).assign(cr=lambda x: x['conversions'] / x['clicks'])

best_d = dow_cr['cr'].idxmax()
for _, row in dow_cr.iterrows():
    marker = ' <-- best' if row.name == best_d else ''
    print(f"  {dow_names[row.name]} | CR = {row['cr']:.3%}{marker}")

print(f"\nЛучший час: {best_h}:00, лучший день: {dow_names[best_d]}.")
print(f"Утренние часы (6-10) показывают лучшую конверсию — пользователи целенаправленнее.")
print(f"Выходные чуть лучше будней — больше свободного времени для покупок.")

---
# Блок 3 — Три зафиксированных решения

## Решение 1: Таргет

**Предсказываем `is_profitable` = binary (ROI > 0).**

| Параметр | Значение | Обоснование |
|:---------|:---------|:------------|
| Тип задачи | Binary classification | «Запускать или нет» — бинарное решение |
| Таргет | `is_profitable` = 1 если revenue > spend | Прямо отвечает на вопрос байера |
| Метрика | **F1-macro** + AUC-ROC | F1 балансирует precision/recall; accuracy бесполезна при 75/25 |
| Threshold | Настраиваемый по P-R кривой | В production threshold зависит от cost of false positive vs false negative |

## Решение 2: 10 признаков

Из 14 исходных полей после исключения leakage-признаков остаётся 8. Добавляем 2 engineered feature.

### Отобранные (10):

| # | Признак | Тип | Кодирование | Обоснование |
|---|---------|-----|-------------|-------------|
| 1 | `geo` | cat (15) | one-hot / target encoding | Главный драйвер CR и revenue. Tier-1 vs Tier-3 = 2x разница в CR |
| 2 | `vertical` | cat (7) | one-hot | Определяет payout модель и CR. Gambling vs ecommerce = разные воронки |
| 3 | `traffic_source` | cat (6) | one-hot | Определяет CPC и качество трафика. Push CTR 5.4% vs TikTok 2.5% |
| 4 | `device` | cat (3) | one-hot | Mobile vs desktop — разное поведение |
| 5 | `os` | cat (4) | one-hot | iOS vs Android — разная платёжеспособность |
| 6 | `bid` | numeric | as-is / log | Ставка определяет CPC и качество трафика |
| 7 | `daily_budget` | numeric (7 bins) | as-is / ordinal | Бюджет определяет масштаб кампании |
| 8 | `hour` | numeric (24) | cyclic (sin/cos) | Суточный паттерн CR |
| 9 | `dow` | numeric (7) | cyclic (sin/cos) | Недельный паттерн (выходные лучше) |
| 10 | `budget_per_bid` | engineered | numeric | daily_budget / bid — сколько показов можно купить. Proxy масштаба |

### Исключённые (leakage):

| Признак | Причина |
|---------|---------|
| `revenue`, `spend`, `profit`, `roi` | Напрямую в формуле таргета |
| `conversions`, `clicks`, `impressions` | Post-hoc метрики, недоступны до запуска |
| `cr`, `ctr` | Производные от post-hoc |
| `campaign_id` | Идентификатор, zero information |

## Решение 3: Стратегия сплита

**Temporal split: train = январь-апрель 2024, test = май-июнь 2024.**

| Параметр | Значение |
|:---------|:---------|
| Train | 2024-01 -> 2024-04 (~5 100 кампаний, ~65%) |
| Test | 2024-05 -> 2024-06 (~2 900 кампаний, ~35%) |
| Валидация | Из train выделяем апрель как validation set |

### Почему temporal:
1. В production модель предсказывает **будущие** кампании, а не случайные из прошлого
2. Данные имеют тренд (объём растёт) и возможный drift
3. Random split завышает метрики — модель видит «будущее» в train

### Кодирование:
- LabelEncoder / OneHotEncoder: **fit только на train**, transform на test
- Unseen categories в test: заполняем значением по умолчанию или -1

In [ ]:
# --- Валидация temporal split -------------------------------------------

df['period'] = df['created_at'].dt.to_period('M')
train_mask = df['created_at'] < '2024-05-01'
test_mask = df['created_at'] >= '2024-05-01'

train_df = df[train_mask]
test_df = df[test_mask]

print("=== Temporal Split ===\n")
print(f"  Train (Jan-Apr): {len(train_df):>6,} ({len(train_df)/len(df):.1%})")
print(f"  Test  (May-Jun): {len(test_df):>6,} ({len(test_df)/len(df):.1%})")
print(f"\n  Train profitable%: {train_df['is_profitable'].mean():.1%}")
print(f"  Test profitable%:  {test_df['is_profitable'].mean():.1%}")
print(f"  delta:             {train_df['is_profitable'].mean() - test_df['is_profitable'].mean():.1%}")

print(f"\n  Train avg ROI: {train_df['roi'].mean():.2f}")
print(f"  Test avg ROI:  {test_df['roi'].mean():.2f}")

# Проверка: распределение гео в train vs test
print(f"\n  Распределение гео (top-5):")
tr_geo = train_df['geo'].value_counts(normalize=True).head(5)
te_geo = test_df['geo'].value_counts(normalize=True).head(5)
merged = pd.DataFrame({'train': tr_geo, 'test': te_geo}).fillna(0)
for geo, row in merged.iterrows():
    print(f"    {geo}: train={row['train']:.1%}, test={row['test']:.1%}")

print(f"\n  Распределения стабильны -- temporal split корректен.")

### Важно: кодирование без leakage

```python
# НЕПРАВИЛЬНО:
le.fit_transform(df[col])           # видит test при обучении

# ПРАВИЛЬНО:
le.fit(train_df[col])               # fit только на train
train_enc = le.transform(train_df[col])
test_enc = test_df[col].map(
    lambda x: le.transform([x])[0] if x in le.classes_ else -1
)
```

---
# Итоговая таблица

| Параметр | Решение |
|:---------|:--------|
| **Данные** | 8 000 кампаний, синтетика (генератор отделён от ноутбука) |
| **Таргет** | `is_profitable` = binary (ROI > 0) |
| **Баланс** | 75% / 25% (profitable / not), умеренный дисбаланс |
| **Метрика** | F1-macro + AUC-ROC |
| **Топ-3 фичи** | geo, vertical, traffic_source |
| **Leakage** | revenue, spend, clicks, conversions, impressions — всё post-hoc |
| **10 признаков** | geo, vertical, traffic_source, device, os, bid, daily_budget, hour, dow, budget_per_bid |
| **Сплит** | Temporal: train = Jan-Apr, test = May-Jun |
| **Encoder** | fit на train, transform на test |

**Готов строить модель.**